# The Poisson distribution
The PMF by hand and with scipy, interval probabilities, mean = variance by simulation, and the binomial-to-Poisson limit.

In [ ]:
import math
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. Exactly 7 questions when 4 are usual (lambda = 4)

In [ ]:
lam, y = 4, 7
# lambda^y * e^(-lambda) / y!
by_hand = lam ** y * math.exp(-lam) / math.factorial(y)
print(lam ** y, round(math.exp(-lam), 4), math.factorial(y))
print('by hand:', round(by_hand, 4), ' scipy:', round(stats.poisson.pmf(y, lam), 4))

## 2. The whole PMF: it adds to 1

In [ ]:
questions = stats.poisson(4)
for k in range(13):
    print(k, round(questions.pmf(k), 4))
# the counts never stop, but the far tail is tiny
print('sum of P(0..50):', questions.pmf(np.arange(51)).sum())

## 3. Probabilities of intervals: add the bars

In [ ]:
print('P(Y >= 7)     =', round(questions.sf(6), 4))     # 1 - P(Y <= 6)
print('P(Y <= 2)     =', round(questions.cdf(2), 4))
print('P(2 <= Y <= 6) =', round(questions.cdf(6) - questions.cdf(1), 4))
print('same by adding bars:', round(questions.pmf(range(2, 7)).sum(), 4))

## 4. Fireflies: 3 flashes per 10 seconds, so 6 per 20 seconds

In [ ]:
rate_per_second = 3 / 10
lam_20 = rate_per_second * 20       # scale the rate to the new interval
print('lambda for 20 s:', lam_20)
print('P(8 flashes in 20 s) =', round(stats.poisson.pmf(8, lam_20), 4))

## 5. Mean and variance are both lambda

In [ ]:
rng = np.random.default_rng(42)     # fixed seed
days = rng.poisson(lam=4, size=100_000)
print('first days:', days[:10])
print('average', days.mean(), ' variance', round(days.var(), 4))
print('scipy: mean', questions.mean(), ' variance', questions.var(), ' std', questions.std())

## 6. Binomial with large n and small p is close to Poisson

In [ ]:
k = np.arange(30)
for n, p in [(10, 0.4), (40, 0.1), (1000, 0.004)]:
    gap = np.abs(stats.binom.pmf(k, n, p) - stats.poisson.pmf(k, n * p)).max()
    print(f'n = {n:5}, p = {p:5}: largest gap {gap:.4f}')

## 7. Try it: drag the slider to change lambda

In [ ]:
y = np.arange(0, 26)
fig = go.Figure()
lams = [0.5, 1, 2, 4, 7, 10, 15]
for lam in lams:
    fig.add_bar(x=y, y=stats.poisson.pmf(y, lam), visible=(lam == 4), name=f'λ = {lam}')
steps = [dict(method='update', label=str(lam), args=[{'visible': [l == lam for l in lams]}]) for lam in lams]
fig.update_layout(sliders=[dict(active=3, currentvalue=dict(prefix='λ = '), steps=steps)],
                  template='simple_white', xaxis_title='count, y', yaxis_title='P(Y = y)')
fig.show()